### Figure 4 C,D,E,F,G, Supplemental Figure 4 A, B

In [ ]:

%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl
from sklearn.linear_model import LinearRegression
from statsmodels.stats.nonparametric import *
from scipy.optimize import curve_fit
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from paths import DATA_DIR, fig_dir
from spyglass.common import Session

In [ ]:
# custom schema
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary
from fig_helpers import *
from plot_figs_summary_metrics import get_all_seg_ahbeh_summary_metric, plot_all_all_seg_ahbeh_metric_switches_only_line_format, get_all_rat_stable_nwb_file_names

### figure setup and load data

In [ ]:
set_figure_defaults()

save_fig = False

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:
# Params
position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

# data loading info
out_path = f'{DATA_DIR}/big_df_pkls/'
today_now = '20240212'

subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

In [ ]:
# load behavior and decoding days of data, crosscheck nwbs
big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')
    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]
print(stable_clusterless_nwbs)

In [ ]:
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}
p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]

# get to stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable[~df_stable[p_rew_cols].eq(df_stable['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
# Filter data to segments of interest
hpd_percent = 50 # or 95
hpd_thresh_cm = 50 # 50
use_abs_ahbeh_thresh = False 
abs_ahbeh_thresh_cm = 10
quantile = .9

big_dfs_firstlast_nonlocal_incljump_grouped = {}
for subject_id in subject_ids:
    big_df = all_rat_big_dfs_stable[subject_id]
    
    # limit to first or last, and hpd and ahbeh restrictions for quality control
    big_df_firstlast = big_df[np.logical_and(
                                    np.logical_or(big_df['is_first_seg_of_trial']==True,
                                                  big_df['is_last_seg_of_trial']==True),
                                    big_df[f'spatial_coverage_{hpd_percent}_hpd']<hpd_thresh_cm,
                                    )]
    if use_abs_ahbeh_thresh:
        big_df_firstlast = big_df_firstlast[big_df_firstlast['abs_ahead_behind_distance']>=abs_ahbeh_thresh_cm]
    
    big_df_firstlast_nonlocal = big_df_firstlast[big_df_firstlast['nonlocal_by_segment']==True]
    
    # find nonlocal stay and switch consistent content
    big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere'] = np.logical_and(
                                                                    big_df_firstlast_nonlocal['nonlocal_by_patch']==False,
                                                                    big_df_firstlast_nonlocal['is_mental_seg_mapped_a_leaf']==True)
        
    big_df_firstlast_nonlocal['is_mental_seg_elsewhere_or_leaf_in_patch'] = ~big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere']
    
    # now do all the groupings via lambda fxns to calc things for first/final seg
    # these calcultaions are only during nonlocal by seg times, not full trial time
    big_df_grouped = big_df_firstlast_nonlocal.groupby(
            by=['nwb_file_name', 'epoch_number', 'trial_number_by_epoch', 'stem_switch', 'stem', 'leaf', 'stemchoice', 'reward', 'is_first_seg_of_trial',
                'trials_from_prior_switch', 'trials_from_next_switch','try_bout_idx','bout_len_new']
        ).apply(
            lambda x_df: pd.Series({
                'prop_elsewhere_vs_neighbor_leaf': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]) / len(x_df),
                'len_elsewhere': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]),
                'len_neighbor_leaf': len(x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]),
                'ahbeh_mean':x_df['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_max':x_df['abs_ahead_behind_distance'].max(),
                'ahbeh_max_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].max(),
                'ahbeh_max_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].max(),
                f'ahbeh_quantile{int(100*quantile)}':x_df['abs_ahead_behind_distance'].quantile(q=quantile),
            })
        ).reset_index()
    big_dfs_firstlast_nonlocal_incljump_grouped[subject_id] = big_df_grouped


In [ ]:
# Make all animal concatenated df, reset idx to get subj col
concatenated_df = pd.concat(big_dfs_firstlast_nonlocal_incljump_grouped.values(), keys=big_dfs_firstlast_nonlocal_incljump_grouped.keys(), names=['subject_id'])
concatenated_df.reset_index(level=0, inplace=True)
concatenated_df.reset_index(drop=True, inplace=True)

# for these analyses baseline subtraction incl switch trial
grouped = concatenated_df.groupby(['subject_id', 'is_first_seg_of_trial'])

# Calculate the mean baseline for each group
mean_baseline = grouped['prop_elsewhere_vs_neighbor_leaf'].transform('mean')
mean_baseline2 = grouped['ahbeh_max'].transform('mean')

# Create the new col
concatenated_df['prop_elsewhere_vs_neighbor_leaf_minus_avg'] = concatenated_df['prop_elsewhere_vs_neighbor_leaf'] - mean_baseline
concatenated_df['ahbeh_max_minus_avg'] = concatenated_df['ahbeh_max'] - mean_baseline2

# baseline subtracted numbers now all not just stay trials

#### F4 C,D,F,G SF4 A,B peri-switch extent

In [ ]:
# Functions

# Define exponential function
def exponential_func(x, a, b):
    return a * np.exp(b * x)

# Function to fit exponential data
def fit_exponential(x, y, is_first_seg_of_trial):
    # Flatten x and y to ensure they are 1D arrays
    x = np.array(x).flatten()
    y = np.array(y).flatten()

    # Estimate initial parameters
    a_initial = 50 #y[0] if x[0] == 0 else y[-1]  # Choose the y value at or near x=0
    b_initial = -.025 if is_first_seg_of_trial else 0.025

    # Fit the data
    params, _ = curve_fit(exponential_func, x, y, p0=[a_initial, b_initial])
    y_fit = exponential_func(x, *params)

    # Create a DataFrame for statsmodels
    df = pd.DataFrame({'x': x, 'y': y})

    # Fit the model using statsmodels for summary
    model = smf.ols('np.log(y) ~ x', data=df).fit()

    return params, y_fit, model

In [ ]:
subject_ids_and_all = subject_ids + ['allrats']

# params
figwidth = TWO_COLUMN/3 
figheight = figwidth
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))
custom_palette_by_rat = {subject_id: next(custom_colors_by_rat) for subject_id in subject_ids}
all_rat_color = 'black'
xmax = 20 # trials
trial_subsets = ['trials_from_next_switch', 'trials_from_prior_switch']
include_switches=True
err_style = 'bars'
markersize=5
ci=95
linewidth=.5
show_shuffle = False
show_linreg=False
offset = 5
metrics = ['ahbeh_max'] #'prop_elsewhere_vs_neighbor_leaf', 'ahbeh_max'] # no content here
show_exp_reg = False
fit_exp_reg=True

In [ ]:

for is_first_seg_of_trial in [True,False]:
    for metric in metrics:
        for subject_id in subject_ids_and_all:    # set up
            if subject_id == 'allrats':
                metric = f'{metric}_minus_avg'

            fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(figwidth,figheight), sharey=True)

            for i,trial_subset in enumerate(trial_subsets):
                concatenated_df_copy = concatenated_df.copy()
                if subject_id != 'allrats':
                    data_df = concatenated_df_copy[concatenated_df_copy['subject_id']==subject_id] #concatenated_df        
                else:
                    data_df = concatenated_df_copy
                    
                data_df = data_df[data_df['is_first_seg_of_trial']==is_first_seg_of_trial]
                
                # Filter data to only what to plot
                if (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]<=-1) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_next_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]<=0) & (data_df[trial_subset]>=-xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == False):
                    data_df = data_df[(data_df[trial_subset]>=0) & (data_df[trial_subset]<=xmax)]
                elif (trial_subset == 'trials_from_prior_switch') & (is_first_seg_of_trial == True):
                    data_df = data_df[(data_df[trial_subset]>=1) & (data_df[trial_subset]<=xmax)]

                if include_switches == False:
                    data_df = data_df[data_df['stem_switch']==False]

            # Plot content or extent variables
            # first plot for CI alpha, then plot bigger markers on top of the data
                sns.lineplot(data=data_df,
                             x=trial_subset, y=metric,
                             markersize=markersize, marker='o', err_style = err_style,
                             color=custom_palette_by_rat[subject_id] if subject_id in subject_ids else all_rat_color,
                             ax=axes[i], ci=ci,  mew=0, alpha=.3,
                             err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                             zorder=99,linewidth=0)
                sns.lineplot(data=data_df,
                             x=trial_subset, y=metric,
                             markersize=markersize, marker='o', err_style = err_style,
                             color=custom_palette_by_rat[subject_id] if subject_id in subject_ids else all_rat_color,
                             ax=axes[i], errorbar=None,  mew=0,
                            #  ax=axes[i], ci=0,  mew=0, #replaced for new version compatability 
                             err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                             zorder=100,linewidth=0)
                
                if show_linreg:
                    x = np.array(data_df[trial_subset]).reshape((-1,1))
                    y = np.array(data_df[metric])
                    #print(x.shape, y.shape)
                    model = LinearRegression().fit(x,y)
                    x_fit = np.linspace(x.min(), x.max(), 100).reshape(-1,1)
                    y_fit = model.predict(x_fit)
                    #r_sq = model.score(x,y)
                    intercept = model.intercept_
                    slope = model.coef_[0]
                    axes[i].plot(x_fit, y_fit, color='black')
                if fit_exp_reg:
                    if metric[0:4]=='ahbe':
                        x = np.array(data_df[trial_subset]).reshape((-1,1))
                        y = np.array(data_df[metric]) 
                        params, y_fit_big, model = fit_exponential(x, y, is_first_seg_of_trial)
                        x_fit = np.linspace(x.min(), x.max(), 100).reshape(-1,1)
                        y_fit = exponential_func(x_fit, *params)
                        print(f"\n\nrat: {subject_id}, firstseg: {is_first_seg_of_trial}, {trial_subset}")
                        print(model.summary())
                        print('pvals: ', model.pvalues, '\n\n')
                        if show_exp_reg:
                            axes[i].plot(x_fit, y_fit, color='black')
                if metric[0:4] == 'prop':
                    if subject_id != 'allrats':
                        axes[i].set_ylim(0,1)
                        axes[0].set_ylabel(f'Proportion of Non-local activity\nalong Switch paths')
                    else:
                        axes[i].set_ylim(-.3,.7)
                        axes[0].set_ylabel(f'Baseline-subtracted\nProprtion of Non-local activity\nalong Switch paths')
                elif metric[0:4] == 'ahbe':
                    if subject_id != 'allrats':
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(0,110)
                        else:
                            axes[i].set_ylim(0,110)
                        axes[0].set_ylabel(f'Max. Non-local Distance')
                    else:
                        if is_first_seg_of_trial:
                            axes[i].set_ylim(-40,70)
                        else:
                            axes[i].set_ylim(-40,70)
                        axes[0].set_ylabel(f'Baseline-subtracted\nMax. Non-local Distance')
                axes[i].set_xlabel('')
            sns.despine(offset=offset)
            # for switches..
            if include_switches == False:
                axes[0].set_xlim(-xmax-1,0)
                axes[1].set_xlim(0,xmax+1)
                axes[0].set_xticks([-xmax,-10,-1], )
                axes[1].set_xticks([1,10,xmax],)
                axes[0].spines['bottom'].set_bounds(-xmax,-1)
                axes[1].spines['bottom'].set_bounds(1,xmax)    
            elif include_switches == True:
                if is_first_seg_of_trial==True:
                    axes[0].set_xlim(-xmax-1,1)
                    axes[1].set_xlim(0,xmax+1)
                    axes[0].set_xticks([-xmax,-10,0], )
                    axes[1].set_xticks([1,10,xmax],)
                    axes[0].spines['bottom'].set_bounds(-xmax,0)
                    axes[1].spines['bottom'].set_bounds(1,xmax)  
                elif is_first_seg_of_trial==False:
                    axes[0].set_xlim(-xmax-1,0)
                    axes[1].set_xlim(-1,xmax+1)
                    axes[0].set_xticks([-xmax,-10,-1], )
                    axes[1].set_xticks([0,10,xmax],)
                    axes[0].spines['bottom'].set_bounds(-xmax,-1)
                    axes[1].spines['bottom'].set_bounds(0,xmax)  
            axes[1].yaxis.set_visible(False)
            axes[1].spines['left'].set_visible(False)

            plt.suptitle(f'Rat {subject_id[0].upper()}, first seg: {is_first_seg_of_trial}', fontsize=6) #, y=1.01)
            fig.text(0.5,-.07, 'Trials since Switch Trial',ha='center')
            fig.subplots_adjust(wspace=.1)

            if save_fig:
                fig_name = f'staysANDswitch_periswitch_OR_expline{show_exp_reg}_{subject_id}_{metric}_firstseg{is_first_seg_of_trial}_inclswitch{include_switches}_err{err_style}_ci{ci}_shuff{show_shuffle}_linreg{show_linreg}_xmax{xmax}_w{figwidth}_h{figheight}_concatc{all_rat_color}_s{markersize}_lw{linewidth}_offset{offset}_ahbehthresh{use_abs_ahbeh_thresh}_ahbehgt{abs_ahbeh_thresh_cm}'
                plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)     

            plt.show()


#### F4E

In [ ]:
summary_metric = 'max'
restrict_to_within_patch = False

all_rat_all_seg_max = {}
for subject_id in subject_ids:
    big_df = all_rat_big_dfs_stable[subject_id]
    grouped_df = get_all_seg_ahbeh_summary_metric(big_df, summary_metric=summary_metric,  restrict_to_within_patch=restrict_to_within_patch)
    all_rat_all_seg_max[subject_id] = grouped_df

In [ ]:
# Max
grouped_data = all_rat_all_seg_max
y_lim = (0,110)
ci = 95
connect_lines = False
figwidth = TWO_COLUMN/4 
figheight = TWO_COLUMN/3

all_nwb_file_names_dict, stable_nwb_file_names_dict = get_all_rat_stable_nwb_file_names(all_rat_big_dfs_stable)
plot_all_all_seg_ahbeh_metric_switches_only_line_format(subject_ids,
                           grouped_data,
                           stable_nwb_file_names_dict, 
                           y_lim=y_lim,
                           ci=ci,
                           fig_path = fig_path, save_fig = save_fig,
                            connect_lines=connect_lines, figwidth=figwidth, figheight=figheight)